# AAC Word Predictor — Merged CHiLDES Corpora (Brown + Providence + Eng-NA-MOR)
This notebook trains a **word-level LSTM** on **multiple CHiLDES corpora** to power on-device, offline word prediction in a React Native AAC app.  
It demonstrates dataset engineering, model design, evaluation, and export to **TensorFlow.js** for mobile integration.

## 0) Environment Setup

In [ ]:
!pip -q install pylangacq tensorflow tensorflowjs matplotlib

## 1) Upload CHiLDES Corpora (ZIP files)
Download the ZIPs locally from https://childes.talkbank.org/access/Eng-NA/ (e.g., **Brown.zip**, **Providence.zip**, **Eng-NA-MOR.zip**) and upload them here.  
You can upload **one** or **all three** — the code will automatically merge them if present.

In [ ]:
from google.colab import files
import os, glob
print('Click the button to select one or more ZIP files...')
uploaded = files.upload()  # select multiple files
print('Uploaded:', list(uploaded.keys()))

# Discover likely corpus zips in the working directory
preferred = ['Brown.zip','Providence.zip','Eng-NA-MOR.zip']
zip_paths = [p for p in preferred if os.path.exists(p)]
if not zip_paths:
    zip_paths = [p for p in glob.glob('*.zip')]
print('Detected ZIPs:', zip_paths)
assert zip_paths, 'No ZIP files detected. Upload at least one CHiLDES ZIP.'

## 2) Load & Inspect Corpora (PyLangAcq)
We read each uploaded ZIP with **PyLangAcq**, extract **child sentences (CHI)**, and merge them.

In [ ]:
import pylangacq, re

all_child_sent_tokens = []  # list[list[str]]
for zp in zip_paths:
    print(f'Loading {zp} ...')
    c = pylangacq.read_chat(zp)
    sents_chi = c.sents(participants='CHI')
    print('  CHI sentences:', len(sents_chi))
    all_child_sent_tokens.extend(sents_chi)

print('TOTAL CHI sentences (merged):', len(all_child_sent_tokens))
print('Example tokenised sentence:', all_child_sent_tokens[0] if all_child_sent_tokens else [])

## 3) Clean & Normalise Tokens, Rebuild Sentences
We keep alphanumeric tokens and apostrophes (drop codes like `xxx`, punctuation), lowercase everything, and keep sentences with at least 4 tokens.

In [ ]:
import re
def keep_token(tok: str) -> bool:
    return bool(re.fullmatch(r"[a-zA-Z0-9']+", tok))
def norm_token(tok: str) -> str:
    return tok.lower()

sentences = []  # list[str]
for toks in all_child_sent_tokens:
    cleaned = [norm_token(t) for t in toks if keep_token(t)]
    if len(cleaned) >= 4:
        sentences.append(' '.join(cleaned))

print('Usable cleaned sentences:', len(sentences))
print('Sample:', sentences[:5])
assert len(sentences) > 0, 'No usable sentences; check uploads or cleaning rules.'

## 4) Tokenise & Build 4-gram Sequences
We cap the vocabulary to keep the mobile model small, and build 4-word context → next-word targets.

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np, random

MAX_VOCAB = 8000           # ✅ adjust to balance size vs. coverage
CONTEXT_LEN = 4            # ✅ context window length expected by the app
MAX_TRAIN_SEQS = 300_000   # ✅ optional cap to keep training time reasonable

tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token='<unk>')
tokenizer.fit_on_texts(sentences)
total_words = min(MAX_VOCAB, len(tokenizer.word_index) + 1)
print('Vocabulary size:', total_words)

input_sequences = []
for line in sentences:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(CONTEXT_LEN, len(token_list)):
        n_gram = token_list[i-CONTEXT_LEN:i+1]
        input_sequences.append(n_gram)

random.shuffle(input_sequences)
if len(input_sequences) > MAX_TRAIN_SEQS:
    input_sequences = input_sequences[:MAX_TRAIN_SEQS]

input_sequences = np.array(input_sequences)
X = input_sequences[:, :-1]
y = input_sequences[:, -1]
print('Total sequences used for training:', len(X))

## 5) Build a Compact LSTM Model
Embedding → LSTM → Dense over vocabulary. Tuned for **on-device** inference.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

EMBED_DIM = 64
LSTM_UNITS = 128
BATCH_SIZE = 128
EPOCHS = 20  # increase to 30+ for better quality if time allows

model = tf.keras.Sequential([
    layers.Embedding(total_words, EMBED_DIM, input_length=CONTEXT_LEN),
    layers.LSTM(LSTM_UNITS),
    layers.Dense(total_words, activation='softmax')
])
model.compile(loss='sparse_categorical_crossentropy', optimizer='rmsprop', metrics=['accuracy'])
model.summary()

## 6) Train

In [ ]:
history = model.fit(X, y, epochs=EPOCHS, batch_size=BATCH_SIZE, validation_split=0.1)

## 7) Evaluation & Analysis

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.title('Accuracy'); plt.xlabel('Epoch'); plt.ylabel('Acc'); plt.legend()
plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss'); plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend()
plt.show()

print('Final Train Acc:', round(history.history['accuracy'][-1], 3))
print('Final Val   Acc:', round(history.history['val_accuracy'][-1], 3))

### Qualitative Predictions

In [ ]:
reverse_word_map = dict(map(reversed, tokenizer.word_index.items()))
def top_n_predictions(words, n=5):
    seq = tokenizer.texts_to_sequences([' '.join(words)])[0]
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    seq = pad_sequences([seq], maxlen=CONTEXT_LEN)
    preds = model.predict(seq, verbose=0)[0]
    import numpy as np
    idxs = np.argsort(preds)[-n:][::-1]
    return [reverse_word_map.get(i, '<unk>') for i in idxs]

samples = [
    ['i','want','to',''],
    ['can','you','help',''],
    ['i','am','feeling','']
]
for s in samples:
    print('Input:', ' '.join(s))
    print('Next word candidates:', top_n_predictions(s))
    print()

## 8) Export to TensorFlow.js + Tokenizer JSON

In [ ]:
import json, tensorflowjs as tfjs, glob
tokenizer_json = tokenizer.to_json()
with open('crowdsourced_aac_tokenizer.json','w') as f:
    f.write(tokenizer_json)
tfjs.converters.save_keras_model(model, 'word_prediction_tfjs')
print('Exported files:', glob.glob('word_prediction_tfjs/*'))

## 9) Download Bundle for App Integration

In [ ]:
!zip -rq aac_word_model_merged.zip word_prediction_tfjs crowdsourced_aac_tokenizer.json
from google.colab import files
files.download('aac_word_model_merged.zip')

## 10) Reflection & Skills Evidence
- **Data engineering:** merging multiple CHiLDES corpora; token filtering; reproducibility.
- **Model design:** compact LSTM tuned for mobile (small embedding, limited vocab, short context).
- **Evaluation:** accuracy/loss curves; qualitative next-word predictions.
- **Integration:** TFJS export + tokenizer JSON for React Native AAC app.
- **Trade-offs:** limiting vocabulary to keep the model small; optional sequence cap to manage training time; bias from child-directed speech acknowledged.